## Create Flag Parameter

In [0]:
dbutils.widgets.text('incremental_flag','0')

In [0]:
incremental_flag = dbutils.widgets.get('incremental_flag')

## Creating Dimension Model

### Fetching relative columns

In [0]:
df_src = spark.sql('''
                   Select Distinct(Date_ID) as Date_ID
                   from parquet.`abfss://silver@slcardatalake.dfs.core.windows.net/carsales/`
                   ''')

In [0]:
%sql
Select 1 as dim_date_key, Date_ID from
  parquet.`abfss://silver@slcardatalake.dfs.core.windows.net/carsales/`
  where 1=0

## dim_model Sink - Initial and Incremental

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_date'):
  df_sink = spark.sql('''
  Select dim_date_key, Date_ID from cars_catalog.gold.dim_date
  ''')
else:
  df_sink= spark.sql('''Select 1 as dim_date_key, Date_ID from
  parquet.`abfss://silver@slcardatalake.dfs.core.windows.net/carsales/`
  where 1=0''')

### Filtering new records and old records

In [0]:
df_filter = df_src.join(df_sink,df_src.Date_ID==df_sink.Date_ID,"left").select(df_src['Date_ID'], df_sink['dim_date_key'])

In [0]:
display(df_filter)

**df_filter_old**

In [0]:
from pyspark.sql.functions import *
df_filter_old = df_filter.filter(col("dim_date_key").isNotNull())  

**df_filter_new**

In [0]:
df_filter_new = df_filter.filter(col("dim_date_key").isNull()).select("Date_ID")

## Create Surrogate Key

**Fetch the max surrogate key from existing table**

In [0]:
if (incremental_flag == '0'):
    max_value = 1
else:
    max_value_df = spark.sql("Select max(dim_date_key) from cars_catalog.gold.dim_date")
    max_value = max_value_df.collect()[0][0] +  1

**Create surrogate key column and add max value to surrogate key**

In [0]:
df_filter_new = df_filter_new.withColumn('dim_date_key', max_value + monotonically_increasing_id())

### Create Final Df - df_filter_old + df_filter_new

In [0]:
df_final = df_filter_new.union(df_filter_old)

## SCD Type 1 (Upsert)

In [0]:
from delta.tables import DeltaTable

In [0]:
#Incremental Run
if spark.catalog.tableExists('cars_catalog.gold.dim_date'):
    delta_tbl = DeltaTable.forPath(spark , 'abfss://gold@slcardatalake.dfs.core.windows.net/dim_date')
    delta_tbl.alias('trg').merge(df_final.alias('src'),'trg.dim_date_key=src.dim_date_key')\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

#Initial run
else:
    df_final.write.format('delta').mode('overwrite')\
        .option('path','abfss://gold@slcardatalake.dfs.core.windows.net/dim_date')\
        .saveAsTable('cars_catalog.gold.dim_date')


In [0]:
%sql
Select * from cars_catalog.gold.dim_date